# Extract DEX prices from Dune

Pull swap-level mid prices for one token pair on one blockchain, from each supported DEX, over a **collection window**, then save one CSV per DEX under `<chain>/`.

All reusable logic lives in the `arblib` package; this notebook only sets parameters and wires the steps together.

In [1]:
# !pip install -r requirements.txt

In [5]:
from arblib.config import SWAP_QUERY_IDS, LIQUIDITY_QUERY_IDS, GAS_QUERY_IDS, USD_PRICE_QUERY_ID, TOKENS, build_collection_params
from arblib.dune_api import make_headers, run_dune_saved_query
from arblib.data_io import save_dataframes
import os
from pathlib import Path


## Parameters

- **CHAIN / tokens** — which market to pull.
- **Collection window** (`START_TS` / `END_TS`, UTC) — note `START_TS` is deliberately *earlier* than the study start used in `arbitrage.ipynb`, so every pool already has a known price to forward-fill from once the study window begins.

In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  
DUNE_API_KEY = os.environ["DUNE_API_KEY"]

current = Path.cwd()
while current.name != 'defi_arbitrage' and current != current.parent:
    current = current.parent
BASE_DIR = current if current.name == 'defi_arbitrage' else Path.cwd()

# --- What to collect ------------------------------------------------
CHAIN  = "ethereum"                 # blockchain name
TOKEN0 = TOKENS[CHAIN]["WETH"]  # base token
TOKEN1 = TOKENS[CHAIN]["USDC"]  # quote token

# --- Collection window (UTC) ---------------------------------------
START_TS = "2025-12-31 15:00:00"
END_TS   = "2025-12-31 16:00:00"




params  = build_collection_params(CHAIN, TOKEN0, TOKEN1, START_TS, END_TS)
headers = make_headers(DUNE_API_KEY)
params

{'start_ts': '2025-12-31 15:00:00',
 'end_ts': '2025-12-31 16:00:00',
 'token0': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'token1': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'chain': 'ethereum'}

## Run the swap queries

Swap-level mid prices, one query per DEX, then **left-joined with that DEX's per-swap gas** (on block / pool / tx hash / event index) so the gas cost rides along with each swap. A DEX not available on `CHAIN` returns an empty DataFrame and is skipped on save. The merged frames are saved under `<chain>/swaps/`.

In [7]:
merge_keys = ["evt_block_number", "pool", "evt_tx_hash", "evt_index"]


df_pancake_swap = run_dune_saved_query(SWAP_QUERY_IDS["pancake"], params, headers, "Pancake")
df_gas_pancake  = run_dune_saved_query(GAS_QUERY_IDS["pancake_gas_per_swap"], params, headers, "Gas pancake")
if not df_pancake_swap.empty and not df_gas_pancake.empty:
    df_pancake_swap = df_pancake_swap.merge(df_gas_pancake, on=merge_keys, how="left")
    df_pancake_swap = df_pancake_swap.sort_values("evt_block_number").reset_index(drop=True)

df_uniswap_swap = run_dune_saved_query(SWAP_QUERY_IDS["uniswap"], params, headers, "Uniswap")
df_gas_uniswap  = run_dune_saved_query(GAS_QUERY_IDS["uniswap_gas_per_swap"], params, headers, "Gas uniswap")
if not df_uniswap_swap.empty and not df_gas_uniswap.empty:
    df_uniswap_swap = df_uniswap_swap.merge(df_gas_uniswap, on=merge_keys, how="left")
    df_uniswap_swap = df_uniswap_swap.sort_values("evt_block_number").reset_index(drop=True)

[Pancake] EXECUTE RESPONSE: {'execution_id': '01KWKY575Q5YBNH3J21PWH8YAY', 'state': 'QUERY_STATE_PENDING'}
[Pancake] STATUS: QUERY_STATE_PENDING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_COMPLETED
[Gas pancake] EXECUTE RESPONSE: {'execution_id': '01KWKY5HFZVMWSN6WTTABM4S6Y', 'state': 'QUERY_STATE_PENDING'}
[Gas pancake] STATUS: QUERY_STATE_PENDING
[Gas pancake] STATUS: QUERY_STATE_EXECUTING
[Gas pancake] STATUS: QUERY_STATE_COMPLETED
[Uniswap] EXECUTE RESPONSE: {'execution_id': '01KWKY5RQPFMZ9YVB2H6EM75HD', 'state': 'QUERY_STATE_PENDING'}
[Uniswap] STATUS: QUERY_STATE_PENDING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_COMPLETED
[Gas uniswap] EXECUTE RESPONSE: {'execution_id': '01KWKY6A7X23NV9H48HW6WVPM5', 'state': 'QUERY_STATE_PENDING'}
[Gas uniswap] STATUS: QUERY_STATE_PEND

In [8]:
swap_dir = os.path.join(BASE_DIR, CHAIN, "swaps")

save_dataframes(
    {
        "df_uniswap_swap.csv": df_uniswap_swap,
        "df_pancake_swap.csv": df_pancake_swap,
    },
    swap_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_uniswap_swap.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_pancake_swap.csv
Done.


## Run the liquidity queries

Mint / burn events per pool, one query per DEX, over the same `params` window. Used downstream to reconstruct the liquidity state at any block.

In [6]:
df_uniswap_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")


if not df_uniswap_liq.empty:
    df_uniswap_liq = df_uniswap_liq.sort_values("evt_block_number").reset_index(drop=True)
if not df_pancake_liq.empty:
    df_pancake_liq = df_pancake_liq.sort_values("evt_block_number").reset_index(drop=True)


[Uniswap liquidity] EXECUTE RESPONSE: {'execution_id': '01KWCSZ62HX3B81GBE087HSTDB', 'state': 'QUERY_STATE_PENDING'}
[Uniswap liquidity] STATUS: None
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_COMPLETED
[Pancake liquidity] EXECUTE RESPONSE: {'execution_id': '01KWCSZGZB639X1TC3838XPRSQ', 'state': 'QUERY_STATE_PENDING'}
[Pancake liquidity] STATUS: QUERY_STATE_PENDING
[Pancake liquidity] STATUS: QUERY_STATE_COMPLETED
[INFO] Pancake liquidity: no data returned


In [7]:
liquidity_dir = os.path.join(BASE_DIR, CHAIN, "liquidity")

save_dataframes(
    {
        "df_uniswap_liq.csv": df_uniswap_liq,
        "df_pancake_liq.csv": df_pancake_liq,
    },
    liquidity_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/liquidity/df_uniswap_liq.csv
Skipped empty or missing dataframe: df_pancake_liq.csv
Done.


## Run the gas query

Per-block base fee + utilization over the same collection window, for the whole chain (not per-DEX). Used downstream to price the gas cost of an arbitrage at any block. The **per-swap** gas is collected with the swaps above and saved under `swaps/`; only this chain-wide gas is saved under `gas/`.

In [8]:
df_gas_chain = run_dune_saved_query(GAS_QUERY_IDS["chain_gas_price"], params, headers, "Gas price")


if not df_gas_chain.empty:
    df_gas_chain = df_gas_chain.sort_values("block_number").reset_index(drop=True)


[Gas price] dropping params not used by query 7748900: ['token0', 'token1']
[Gas price] EXECUTE RESPONSE: {'execution_id': '01KWCSZN5D05ST4DZQZ6Q4XKDK', 'state': 'QUERY_STATE_PENDING'}
[Gas price] STATUS: QUERY_STATE_EXECUTING
[Gas price] STATUS: QUERY_STATE_COMPLETED


In [9]:
gas_dir = os.path.join(BASE_DIR, CHAIN, "gas")

save_dataframes(
    {
        "chain_gas_price.csv": df_gas_chain,
    },
    gas_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/gas/chain_gas_price.csv
Done.


## Run the USD token-price query

Hourly USD price for each token of the pair (from `prices.hour`), over the same
collection window. Used downstream to convert token amounts / prices into USD.
Saved under `<chain>/prices/`.

In [3]:
USD_token_prices = run_dune_saved_query(USD_PRICE_QUERY_ID["USD_price"], params, headers, "USD token prices")

if not USD_token_prices.empty:
    USD_token_prices = USD_token_prices.sort_values(["hour", "contract_address"]).reset_index(drop=True)
USD_token_prices

[USD token prices] EXECUTE RESPONSE: {'execution_id': '01KWHTBNPWA1RBWRDHTEDQ6BD9', 'state': 'QUERY_STATE_PENDING'}
[USD token prices] STATUS: QUERY_STATE_EXECUTING
[USD token prices] STATUS: QUERY_STATE_COMPLETED


,contract_address,hour,price,symbol
0,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,2025-12-31 15:00:00.000 UTC,1.00029,USDC
1,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,2025-12-31 15:00:00.000 UTC,2979.47000,WETH


In [4]:
prices_dir = os.path.join(BASE_DIR, CHAIN, "prices")

save_dataframes(
    {
        "USD_token_prices.csv": USD_token_prices,
    },
    prices_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/prices/USD_token_prices.csv
Done.
